In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import os 
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK SC']
plt.rcParams['axes.unicode_minus'] = False
import os 

from sklearn.metrics import normalized_mutual_info_score as NMI
from tqdm import tqdm
import json 
from multiprocessing import Pool
from functools import partial

## 实现王老师的傻逼消融 

1. 基础全局RF 
2. local RF + 全局RF 
3. local RF + 冷启动

### local RF 无冷启动

In [40]:
'''数据准备''' 


species_id = 'ZWS'
dir_path = f'../phd_thesis/103_leave/{species_id}'
feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'),index_col = 0)
summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'),index_col = 0)
assert (meta_df.index == feature_df.index).all()
assert (summary_df.index == feature_df.columns).all()

species_list = meta_df.index.values
species_label = meta_df.label.values 
species_chinese = meta_df['species_chinese'].values


In [87]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from multiprocessing import Pool
from functools import partial
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score

# --- 1. 定义单个物种的处理函数 ---
def process_single_species(species_id):
    """
    针对单个物种进行 1-30 个特征数量的消融实验
    实验逻辑：
    - 如果是鲸目/翼手目：仅使用同目的其他物种作为训练集（局部模型）
    - 如果是其他目：使用除自身外的全量物种作为训练集（全局模型）
    """
    method_train_summary = {}
    method_test_summary = {}
    
    # 路径请确保正确
    dir_path = f'../phd_thesis/103_leave/{species_id}'
    
    try:
        # 读取数据
        feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col=0)
        meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col=0)
        
        # 基础校验
        if species_id not in meta_df.index:
            return None

        # 获取当前物种的目和真实标签
        current_order = meta_df.loc[species_id, 'order_chinese_new']
        true_label = int(meta_df.loc[species_id, 'label'])
        
        # 循环测试不同特征数量 (1-30)
        for feature_num in range(1, 31):
            # 特征切片与编码
            raw_features = feature_df.iloc[:, :feature_num]
            
            # 使用 OrdinalEncoder 将氨基酸字符串转为整数
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            X_all = encoder.fit_transform(raw_features)
            # 随机森林不处理-1，统一转为0
            X_all = np.where(X_all == -1, 0, X_all)
            
            X_df = pd.DataFrame(X_all, index=feature_df.index)
            y_series = meta_df['label']

            # --- 消融实验核心逻辑 ---
            if current_order in ['鲸目', '翼手目','啮齿目']:
                # 局部训练集：同目且非自身
                train_indices = [
                    s for s in meta_df.index 
                    if (meta_df.loc[s, 'order_chinese_new'] == current_order) and (s != species_id)
                ]
            else:
                # 全局训练集：非自身的所有物种
                train_indices = [s for s in meta_df.index if s != species_id]
            
            # 如果训练集为空（极端情况），跳过
            if not train_indices:
                continue

            X_train = X_df.loc[train_indices]
            y_train = y_series.loc[train_indices]

            # 训练随机森林 (n_jobs=1，避免与多进程Pool冲突)
            model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
            model.fit(X_train, y_train)

            # 评估
            train_acc = accuracy_score(y_train, model.predict(X_train))
            target_pred = model.predict(X_df.loc[[species_id]])[0]

            method_train_summary[feature_num] = train_acc
            method_test_summary[feature_num] = int(target_pred)

        return {
            'species_id': species_id,
            'order': current_order,
            'train_acc': method_train_summary,
            'test_pred': method_test_summary,
            'true_label': true_label
        }
        
    except Exception as e:
        print(f"Error processing {species_id}: {str(e)}")
        return None

# --- 2. 主程序入口 ---
if __name__ == '__main__':
    
    MAX_PROCESSES = 64

    '''数据准备''' 


    species_id = 'ZWS'
    dir_path = f'../phd_thesis/103_leave/{species_id}'
    feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
    meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'),index_col = 0)
    summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'),index_col = 0)
    assert (meta_df.index == feature_df.index).all()
    assert (summary_df.index == feature_df.columns).all()

    species_list = meta_df.index.values
    species_label = meta_df.label.values 
    species_chinese = meta_df['species_chinese'].values
    
    with Pool(processes=MAX_PROCESSES) as pool:
        # 修正：直接传递函数名，不再使用 partial 传递多余参数
        results = pool.map(process_single_species, species_list)

    valid_results = [r for r in results if r is not None]

    # --- 3. 结果汇总与导出 ---
    final_records = []
    for r in valid_results:
        sid = r['species_id']
        order = r['order']
        true_l = r['true_label']
        for f_num in r['test_pred'].keys():
            final_records.append({
                'species_id': sid,
                'feature_num': f_num,
                'pred_acc': int(r['test_pred'][f_num] == true_l)
            })

    df_res = pd.DataFrame(final_records)
    
    # 按照物种和特征数排序
    df_res = df_res.sort_values(by=['species_id', 'feature_num'])
    
    df_res = df_res.pivot(index = 'species_id', columns = 'feature_num')
    df_res.to_csv('local_RF.csv', index=False)
    best_id = df_res.mean().argmax()
    print(f'best feature num {best_id+1}, acc {df_res.mean().iloc[best_id]}')
    err_case = df_res.index[df_res.iloc[:,best_id] == 0]
    print(err_case)


best feature num 2, acc 0.9615384615384616
Index(['Condylura_cristata', 'Echinops_telfairi', 'Physeter_catodon', 'ZWS'], dtype='object', name='species_id')


In [88]:
meta_df.loc[err_case,]


,split,label,order,order_scientific,species_chinese,order_chinese,order_chinese_new
species_id,,,,,,,
Condylura_cristata,diurnal,0.0,other,Eulipotyphla,星鼻鼹,真盲缺目,真盲缺目
Echinops_telfairi,pair_1,1.0,madaowei,Afrosoricida,小马岛猬,非洲鼩目,非洲兽目
Physeter_catodon,pair_2,1.0,whale,Cetacea,抹香鲸,鲸目,鲸目
ZWS,pred,1.0,pred,Rodentia,猪尾鼠,啮齿目,啮齿目


### 全局RF

In [85]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from multiprocessing import Pool
from functools import partial
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import accuracy_score

# --- 1. 定义单个物种的处理函数 ---
def process_single_species(species_id):
    """
    针对单个物种进行 1-30 个特征数量的消融实验
    实验逻辑：
    - 如果是鲸目/翼手目：仅使用同目的其他物种作为训练集（局部模型）
    - 如果是其他目：使用除自身外的全量物种作为训练集（全局模型）
    """
    method_train_summary = {}
    method_test_summary = {}
    
    # 路径请确保正确
    dir_path = f'../phd_thesis/103_leave/{species_id}'
    
    try:
        # 读取数据
        feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col=0)
        meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col=0)
        
        # 基础校验
        if species_id not in meta_df.index:
            return None

        # 获取当前物种的目和真实标签
        current_order = meta_df.loc[species_id, 'order_chinese_new']
        true_label = int(meta_df.loc[species_id, 'label'])
        
        # 循环测试不同特征数量 (1-30)
        for feature_num in range(1, 31):
            # 特征切片与编码
            raw_features = feature_df.iloc[:, :feature_num]
            
            # 使用 OrdinalEncoder 将氨基酸字符串转为整数
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            X_all = encoder.fit_transform(raw_features)
            # 随机森林不处理-1，统一转为0
            X_all = np.where(X_all == -1, 0, X_all)
            
            X_df = pd.DataFrame(X_all, index=feature_df.index)
            y_series = meta_df['label']

            # # --- 消融实验核心逻辑 ---
            # if current_order in ['鲸目', '翼手目','啮齿目']:
            #     # 局部训练集：同目且非自身
            #     train_indices = [
            #         s for s in meta_df.index 
            #         if (meta_df.loc[s, 'order_chinese_new'] == current_order) and (s != species_id)
            #     ]
            # else:
            #     # 全局训练集：非自身的所有物种
            train_indices = [s for s in meta_df.index if s != species_id]
            
            # 如果训练集为空（极端情况），跳过
            if not train_indices:
                continue

            X_train = X_df.loc[train_indices]
            y_train = y_series.loc[train_indices]

            # 训练随机森林 (n_jobs=1，避免与多进程Pool冲突)
            model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
            model.fit(X_train, y_train)

            # 评估
            train_acc = accuracy_score(y_train, model.predict(X_train))
            target_pred = model.predict(X_df.loc[[species_id]])[0]

            method_train_summary[feature_num] = train_acc
            method_test_summary[feature_num] = int(target_pred)

        return {
            'species_id': species_id,
            'order': current_order,
            'train_acc': method_train_summary,
            'test_pred': method_test_summary,
            'true_label': true_label
        }
        
    except Exception as e:
        print(f"Error processing {species_id}: {str(e)}")
        return None

# --- 2. 主程序入口 ---
if __name__ == '__main__':
    
    MAX_PROCESSES = 64

    '''数据准备''' 


    species_id = 'ZWS'
    dir_path = f'../phd_thesis/103_leave/{species_id}'
    feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
    meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'),index_col = 0)
    summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'),index_col = 0)
    assert (meta_df.index == feature_df.index).all()
    assert (summary_df.index == feature_df.columns).all()

    species_list = meta_df.index.values
    species_label = meta_df.label.values 
    species_chinese = meta_df['species_chinese'].values
    
    with Pool(processes=MAX_PROCESSES) as pool:
        # 修正：直接传递函数名，不再使用 partial 传递多余参数
        results = pool.map(process_single_species, species_list)

    valid_results = [r for r in results if r is not None]

    # --- 3. 结果汇总与导出 ---
    final_records = []
    for r in valid_results:
        sid = r['species_id']
        order = r['order']
        true_l = r['true_label']
        for f_num in r['test_pred'].keys():
            final_records.append({
                'species_id': sid,
                'feature_num': f_num,
                'pred_acc': int(r['test_pred'][f_num] == true_l)
            })

    df_res = pd.DataFrame(final_records)
    
    # 按照物种和特征数排序
    df_res = df_res.sort_values(by=['species_id', 'feature_num'])
    
    df_res = df_res.pivot(index = 'species_id', columns = 'feature_num')
    df_res.to_csv('basic_RF.csv', index=False)
    best_id = df_res.mean().argmax()
    print(f'best feature num {best_id+1}, acc {df_res.mean().iloc[best_id]}')
    err_case = df_res.index[df_res.iloc[:,best_id] == 0]
    print(err_case)
    meta_df.loc[err_case,]


best feature num 2, acc 0.9519230769230769
Index(['Condylura_cristata', 'Echinops_telfairi', 'Hipposideros_armiger',
       'Physeter_catodon', 'ZWS'],
      dtype='object', name='species_id')


In [86]:
meta_df.loc[err_case,]


,split,label,order,order_scientific,species_chinese,order_chinese,order_chinese_new
species_id,,,,,,,
Condylura_cristata,diurnal,0.0,other,Eulipotyphla,星鼻鼹,真盲缺目,真盲缺目
Echinops_telfairi,pair_1,1.0,madaowei,Afrosoricida,小马岛猬,非洲鼩目,非洲兽目
Hipposideros_armiger,no_use,1.0,bat,Chiroptera,大蹄蝠,翼手目,翼手目
Physeter_catodon,pair_2,1.0,whale,Cetacea,抹香鲸,鲸目,鲸目
ZWS,pred,1.0,pred,Rodentia,猪尾鼠,啮齿目,啮齿目


### LOCAL RF + 冷启动

In [176]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
from functools import partial
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

# 设置最大进程数为 64
MAX_PROCESSES = 64

def process_species(species_id, top_k):
    """
    使用 RandomForest 随机森林模型计算物种的预测概率，包含冷启动策略
    返回 (species_id, prob_list) 元组，prob_list 包含每个 feature_num 下预测为 label=1 的概率
    """

    dir_path = f'/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/103_leave/{species_id}'
    feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
    meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col = 0)
    summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'), index_col = 0)

    # if species_id == 'Sorex_araneus':
    # score_map = {5: 1, 4: 0.9, 3: 0.8, 2: 0.5, 1:0.1,0: 0}
    ## 重新打分

    score_map_4 = {4: 1, 3:0.95, 2: 0.90, 1:0.1, 0:0}
    score_map_5 = {5: 1, 4: 0.95, 3: 0.75, 2: 0.5, 1:0.1,0: 0}

    if summary_df.iloc[:,:5].sum(axis = 0).min() == 0:
        summary_df.loc[:,'cover_score'] = summary_df.loc[:,'eco_cover'].map(score_map_4)
    else: 
        summary_df.loc[:,'cover_score'] = summary_df.loc[:,'eco_cover'].map(score_map_5)
    summary_df.loc[:,'score'] = summary_df.loc[:,'cover_score'] * summary_df.loc[:,'NMI']

    summary_df = summary_df.sort_values(by = 'score', ascending=False)
    feature_df = feature_df.loc[:,summary_df.index]
    
    assert (meta_df.index == feature_df.index).all()
    assert (summary_df.index == feature_df.columns).all()

    species_order = meta_df.loc[species_id,'order_chinese_new']
    idx = summary_df['Eco_Mode'].values != '-'
    feature_df = feature_df.loc[:,idx]
    summary_df = summary_df.loc[idx,:]


    ### 构建 ref_list（训练集物种）
    if species_order == '鲸目':
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'偶蹄目']), :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '翼手目':
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'啮齿目']), :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '真盲缺目':
        ref_list = ['Condylura_cristata',"Ceratotherium_simum_simum","Equus_asinus","Equus_quagga",]
        if species_id in ref_list:
            ref_list.remove(species_id)
    # elif species_order == '真盲缺目':
    #     tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'真盲缺目']), :]
    #     ref_list = tmp_meta_df.index.to_list()
    #     if species_id in ref_list:
    #         ref_list.remove(species_id)

    elif species_order == '啮齿目': 
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].values == species_order, :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)
            if species_id != 'ZWS':
                ref_list.remove('ZWS') # 啮齿目中唯一的非回声物种

    elif species_order == '攀鼩目':
        ref_list = ['Propithecus_coquereli','Gorilla_gorilla_gorilla', 'Pan_paniscus','Pan_troglodytes','Marmota_monax']
    elif species_order == '兔形目':
        ref_list = ['Propithecus_coquereli','Gorilla_gorilla_gorilla', 'Pan_paniscus','Ochotona_princeps', 'Ochotona_curzoniae']
        if species_id in ref_list:
            ref_list.remove(species_id)
    else:
        # 其他目：食肉目 奇蹄目 偶蹄目 非洲兽目 灵长目
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].values == species_order, :]
        tmp_meta_df = tmp_meta_df.loc[tmp_meta_df['label'].values == 0, :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)
    
    train_species = ref_list.copy()
    
    prob_list = []
    '''鲸鱼 翼手使用机器学习模型,使用top10 feature'''
    if species_order in ['翼手目','鲸目',]:
        for feature_num in range(1, 11):
            train_feature = feature_df.loc[train_species, :].iloc[:, :feature_num]
            train_label   = meta_df.loc[train_species, 'label'].values
            target_feature = feature_df.loc[[species_id], :].iloc[:, :feature_num]
            
            # ---- Encoding：使用 OrdinalEncoder 转换为整数 ----
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoder.fit(train_feature)
            
            X_train = encoder.transform(train_feature)
            X_target = encoder.transform(target_feature)
            
            # RandomForest 可以处理浮点数，但为了保持一致性，将 -1（未知）替换为 0
            X_train_rf = np.where(X_train == -1, 0, X_train)
            X_target_rf = np.where(X_target == -1, 0, X_target)

            model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
            model.fit(X_train_rf, train_label)
            prob_pred = model.predict_proba(X_target_rf)[0, 1]  # 取 label=1 的概率
            prob_list.append(prob_pred)
    else:
        prob_list = []
        for feature_num in range(1, top_k):
            all_feature = feature_df.iloc[:, :feature_num]
            
            eco_mu = summary_df['Eco_Mode'].values[:feature_num]
            
            pred_count = (all_feature.loc[species_id].values == eco_mu).sum()
            ref_count = (all_feature.loc[ref_list,:].values == eco_mu).sum(1)
            ref_max = ref_count.max()
            
            if pred_count < int(0.07*feature_num):
                pred_count = 0            
            score = (0.01+ pred_count) / (0.01+ ref_max)
            score = min(score, 10) 
            prob = score / (1+score) - 0.01 ## 要求更强的概率
            prob_list.append(prob)
    
    return (species_id, prob_list)

In [177]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import get_cmap

dir_path = '/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/103_leave/ZWS'
meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col=0)
meta_df = meta_df.loc[meta_df['label'].values != 2,:]
key_species_list = meta_df.index.to_list()

print(f"总共有 {len(key_species_list)} 个物种需要处理")
print(f"使用 {MAX_PROCESSES} 个进程并行处理")

top_k = 501  # 特征数量范围 1-1000
res_dic = {}

with Pool(processes=MAX_PROCESSES) as pool:
    process_func = partial(process_species, top_k=top_k)
    for species_id, prob_list in tqdm(pool.imap_unordered(process_func, key_species_list), 
                                       total=len(key_species_list), 
                                       desc="Processing species"):
        res_dic[species_id] = prob_list
        
pred_dic = {}
for ele in res_dic: 
    pred_dic[ele] = (np.array(res_dic[ele]).mean() > 0.5).astype(int)
pred_res = pd.Series(pred_dic) 
species_label = meta_df.loc[pred_res.index, 'label'] 

pred_res = pd.DataFrame([pred_res, species_label], index = ['pred', 'label']).T
print(f"acc {(pred_res['pred'] == pred_res['label']).sum()/pred_res.shape[0]}")

err_case = pred_res.index[pred_res['pred'] != pred_res['label']]
print(err_case)
meta_df.loc[err_case,]

总共有 104 个物种需要处理
使用 64 个进程并行处理


Processing species: 100%|██████████| 104/104 [00:02<00:00, 51.04it/s]


acc 0.9615384615384616
Index(['Chrysochloris_asiatica', 'Condylura_cristata', 'Sorex_araneus',
       'Physeter_catodon'],
      dtype='object')


,split,label,order,order_scientific,species_chinese,order_chinese,order_chinese_new
Chrysochloris_asiatica,control_1,0.0,madaowei,Afrosoricida,开普金鼹,非洲鼩目,非洲兽目
Condylura_cristata,diurnal,0.0,other,Eulipotyphla,星鼻鼹,真盲缺目,真盲缺目
Sorex_araneus,no_use,1.0,shichong,Eulipotyphla,普通鼩鼱,真盲缺目,真盲缺目
Physeter_catodon,pair_2,1.0,whale,Cetacea,抹香鲸,鲸目,鲸目


### LOCAL RF + 冷启动 + 随机特征

In [143]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
from functools import partial
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

# 设置最大进程数为 64
MAX_PROCESSES = 64

def process_species(species_id, top_k):
    """
    使用 RandomForest 随机森林模型计算物种的预测概率，包含冷启动策略
    返回 (species_id, prob_list) 元组，prob_list 包含每个 feature_num 下预测为 label=1 的概率
    """

    dir_path = f'/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/103_leave/{species_id}'
    feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
    meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col = 0)
    summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'), index_col = 0)

    # if species_id == 'Sorex_araneus':
    # score_map = {5: 1, 4: 0.9, 3: 0.8, 2: 0.5, 1:0.1,0: 0}
    ## 重新打分

    score_map_4 = {4: 1, 3:0.95, 2: 0.9, 1:0.1, 0:0}
    score_map_5 = {5: 1, 4: 0.90, 3: 0.75, 2: 0.5, 1:0.1,0: 0}

    if summary_df.iloc[:,:5].sum(axis = 0).min() == 0:
        summary_df.loc[:,'cover_score'] = summary_df.loc[:,'eco_cover'].map(score_map_4)
    else: 
        summary_df.loc[:,'cover_score'] = summary_df.loc[:,'eco_cover'].map(score_map_5)
    summary_df.loc[:,'score'] = summary_df.loc[:,'cover_score'] * summary_df.loc[:,'NMI']

    summary_df = summary_df.sort_values(by = 'score', ascending=False)
    N = summary_df.shape[0]
    rand_feature = np.random.choice(np.arange(N), 5000, replace = False)
    summary_df = summary_df.iloc[rand_feature,:]
    
    feature_df = feature_df.loc[:,summary_df.index]
    
    assert (meta_df.index == feature_df.index).all()
    assert (summary_df.index == feature_df.columns).all()

    species_order = meta_df.loc[species_id,'order_chinese_new']
    idx = summary_df['Eco_Mode'].values != '-'
    feature_df = feature_df.loc[:,idx]
    summary_df = summary_df.loc[idx,:]


    ### 构建 ref_list（训练集物种）
    if species_order == '鲸目':
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'偶蹄目']), :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '翼手目':
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'啮齿目']), :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '真盲缺目':
        ref_list = ['Condylura_cristata',"Ceratotherium_simum_simum","Equus_asinus","Equus_quagga",]
        if species_id in ref_list:
            ref_list.remove(species_id)
    # elif species_order == '真盲缺目':
    #     tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'真盲缺目']), :]
    #     ref_list = tmp_meta_df.index.to_list()
    #     if species_id in ref_list:
    #         ref_list.remove(species_id)

    elif species_order == '啮齿目': 
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].values == species_order, :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)
            if species_id != 'ZWS':
                ref_list.remove('ZWS') # 啮齿目中唯一的非回声物种

    elif species_order == '攀鼩目':
        ref_list = ['Propithecus_coquereli','Gorilla_gorilla_gorilla', 'Pan_paniscus','Pan_troglodytes','Marmota_monax']
    elif species_order == '兔形目':
        ref_list = ['Propithecus_coquereli','Gorilla_gorilla_gorilla', 'Pan_paniscus','Ochotona_princeps', 'Ochotona_curzoniae']
        if species_id in ref_list:
            ref_list.remove(species_id)
    else:
        # 其他目：食肉目 奇蹄目 偶蹄目 非洲兽目 灵长目
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].values == species_order, :]
        tmp_meta_df = tmp_meta_df.loc[tmp_meta_df['label'].values == 0, :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)
    
    train_species = ref_list.copy()
    
    prob_list = []
    '''鲸鱼 翼手使用机器学习模型,使用top10 feature'''
    if species_order in ['翼手目','鲸目',]:
        for feature_num in range(1, 11):
            train_feature = feature_df.loc[train_species, :].iloc[:, :feature_num]
            train_label   = meta_df.loc[train_species, 'label'].values
            target_feature = feature_df.loc[[species_id], :].iloc[:, :feature_num]
            
            # ---- Encoding：使用 OrdinalEncoder 转换为整数 ----
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoder.fit(train_feature)
            
            X_train = encoder.transform(train_feature)
            X_target = encoder.transform(target_feature)
            
            # RandomForest 可以处理浮点数，但为了保持一致性，将 -1（未知）替换为 0
            X_train_rf = np.where(X_train == -1, 0, X_train)
            X_target_rf = np.where(X_target == -1, 0, X_target)

            model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
            model.fit(X_train_rf, train_label)
            prob_pred = model.predict_proba(X_target_rf)[0, 1]  # 取 label=1 的概率
            prob_list.append(prob_pred)
    else:
        prob_list = []
        for feature_num in range(1, top_k):
            all_feature = feature_df.iloc[:, :feature_num]
            
            eco_mu = summary_df['Eco_Mode'].values[:feature_num]
            
            pred_count = (all_feature.loc[species_id].values == eco_mu).sum()
            ref_count = (all_feature.loc[ref_list,:].values == eco_mu).sum(1)
            ref_max = ref_count.max()
            
            if pred_count < int(0.07*feature_num):
                pred_count = 0            
            score = (0.01+ pred_count) / (0.01+ ref_max)
            score = min(score, 10) 
            prob = score / (1+score) - 0.01 ## 要求更强的概率
            prob_list.append(prob)
    
    return (species_id, prob_list)

In [144]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import get_cmap

dir_path = '/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/103_leave/ZWS'
meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col=0)
meta_df = meta_df.loc[meta_df['label'].values != 2,:]
key_species_list = meta_df.index.to_list()

print(f"总共有 {len(key_species_list)} 个物种需要处理")
print(f"使用 {MAX_PROCESSES} 个进程并行处理")

top_k = 501  # 特征数量范围 1-1000
res_dic = {}

with Pool(processes=MAX_PROCESSES) as pool:
    process_func = partial(process_species, top_k=top_k)
    for species_id, prob_list in tqdm(pool.imap_unordered(process_func, key_species_list), 
                                       total=len(key_species_list), 
                                       desc="Processing species"):
        res_dic[species_id] = prob_list
        
pred_dic = {}
for ele in res_dic: 
    pred_dic[ele] = (np.array(res_dic[ele]).mean() > 0.5).astype(int)
pred_res = pd.Series(pred_dic) 
species_label = meta_df.loc[pred_res.index, 'label'] 

pred_res = pd.DataFrame([pred_res, species_label], index = ['pred', 'label']).T
print(f"acc {(pred_res['pred'] == pred_res['label']).sum()/pred_res.shape[0]}")

err_case = pred_res.index[pred_res['pred'] != pred_res['label']]
print(err_case)

总共有 104 个物种需要处理
使用 64 个进程并行处理


Processing species: 100%|██████████| 104/104 [00:01<00:00, 52.03it/s]


acc 0.9038461538461539
Index(['Echinops_telfairi', 'Elephas_maximus_indicus', 'Sorex_araneus',
       'Propithecus_coquereli', 'shrew_mole', 'ZWS', 'Physeter_catodon',
       'Balaenoptera_acutorostrata_scammoni', 'Rhinolophus_ferrumequinum',
       'Rousettus_aegyptiacus'],
      dtype='object')


In [145]:
meta_df.loc[err_case,]

,split,label,order,order_scientific,species_chinese,order_chinese,order_chinese_new
Echinops_telfairi,pair_1,1.0,madaowei,Afrosoricida,小马岛猬,非洲鼩目,非洲兽目
Elephas_maximus_indicus,diurnal,0.0,other,Proboscidea,亚洲象,长鼻目,非洲兽目
Sorex_araneus,no_use,1.0,shichong,Eulipotyphla,普通鼩鼱,真盲缺目,真盲缺目
Propithecus_coquereli,diurnal,0.0,other,Primates,科氏冕狐猴,灵长目,灵长目
shrew_mole,pred,1.0,pred,Eulipotyphla,鼩鼹,真盲缺目,真盲缺目
ZWS,pred,1.0,pred,Rodentia,猪尾鼠,啮齿目,啮齿目
Physeter_catodon,pair_2,1.0,whale,Cetacea,抹香鲸,鲸目,鲸目
Balaenoptera_acutorostrata_scammoni,control_2,0.0,whale,Cetacea,小须鲸,鲸目,鲸目
Rhinolophus_ferrumequinum,no_use,1.0,bat,Chiroptera,大菊头蝠,翼手目,翼手目
Rousettus_aegyptiacus,no_use,1.0,bat_new,Chiroptera,埃及果蝠,翼手目,翼手目


### LOCAL RF + 冷启动 + NMI

In [168]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
from functools import partial
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

# 设置最大进程数为 64
MAX_PROCESSES = 64

def process_species(species_id, top_k):
    """
    使用 RandomForest 随机森林模型计算物种的预测概率，包含冷启动策略
    返回 (species_id, prob_list) 元组，prob_list 包含每个 feature_num 下预测为 label=1 的概率
    """

    dir_path = f'/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/103_leave/{species_id}'
    feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
    meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col = 0)
    summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'), index_col = 0)

    # if species_id == 'Sorex_araneus':
    # score_map = {5: 1, 4: 0.9, 3: 0.8, 2: 0.5, 1:0.1,0: 0}
    ## 重新打分

    score_map_4 = {4: 1, 3:0.95, 2: 0.9, 1:0.1, 0:0}
    score_map_5 = {5: 1, 4: 0.90, 3: 0.75, 2: 0.5, 1:0.1,0: 0}

    if summary_df.iloc[:,:5].sum(axis = 0).min() == 0:
        summary_df.loc[:,'cover_score'] = summary_df.loc[:,'eco_cover'].map(score_map_4)
    else: 
        summary_df.loc[:,'cover_score'] = summary_df.loc[:,'eco_cover'].map(score_map_5)
    summary_df.loc[:,'score'] = summary_df.loc[:,'cover_score'] * summary_df.loc[:,'NMI']

    summary_df = summary_df.sort_values(by = 'NMI', ascending=False)
    
    feature_df = feature_df.loc[:,summary_df.index]
    
    assert (meta_df.index == feature_df.index).all()
    assert (summary_df.index == feature_df.columns).all()

    species_order = meta_df.loc[species_id,'order_chinese_new']
    idx = summary_df['Eco_Mode'].values != '-'
    feature_df = feature_df.loc[:,idx]
    summary_df = summary_df.loc[idx,:]


    ### 构建 ref_list（训练集物种）
    if species_order == '鲸目':
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'偶蹄目']), :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '翼手目':
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'啮齿目']), :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '真盲缺目':
        ref_list = ['Condylura_cristata',"Ceratotherium_simum_simum","Equus_asinus","Equus_quagga",]
        if species_id in ref_list:
            ref_list.remove(species_id)
    # elif species_order == '真盲缺目':
    #     tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'真盲缺目']), :]
    #     ref_list = tmp_meta_df.index.to_list()
    #     if species_id in ref_list:
    #         ref_list.remove(species_id)

    elif species_order == '啮齿目': 
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].values == species_order, :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)
            if species_id != 'ZWS':
                ref_list.remove('ZWS') # 啮齿目中唯一的非回声物种

    elif species_order == '攀鼩目':
        ref_list = ['Propithecus_coquereli','Gorilla_gorilla_gorilla', 'Pan_paniscus','Pan_troglodytes','Marmota_monax']
    elif species_order == '兔形目':
        ref_list = ['Propithecus_coquereli','Gorilla_gorilla_gorilla', 'Pan_paniscus','Ochotona_princeps', 'Ochotona_curzoniae']
        if species_id in ref_list:
            ref_list.remove(species_id)
    else:
        # 其他目：食肉目 奇蹄目 偶蹄目 非洲兽目 灵长目
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].values == species_order, :]
        tmp_meta_df = tmp_meta_df.loc[tmp_meta_df['label'].values == 0, :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)
    
    train_species = ref_list.copy()
    
    prob_list = []
    '''鲸鱼 翼手使用机器学习模型,使用top10 feature'''
    if species_order in ['翼手目','鲸目',]:
        for feature_num in range(1, 11):
            train_feature = feature_df.loc[train_species, :].iloc[:, :feature_num]
            train_label   = meta_df.loc[train_species, 'label'].values
            target_feature = feature_df.loc[[species_id], :].iloc[:, :feature_num]
            
            # ---- Encoding：使用 OrdinalEncoder 转换为整数 ----
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoder.fit(train_feature)
            
            X_train = encoder.transform(train_feature)
            X_target = encoder.transform(target_feature)
            
            # RandomForest 可以处理浮点数，但为了保持一致性，将 -1（未知）替换为 0
            X_train_rf = np.where(X_train == -1, 0, X_train)
            X_target_rf = np.where(X_target == -1, 0, X_target)

            model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
            model.fit(X_train_rf, train_label)
            prob_pred = model.predict_proba(X_target_rf)[0, 1]  # 取 label=1 的概率
            prob_list.append(prob_pred)
    else:
        prob_list = []
        for feature_num in range(1, top_k):
            all_feature = feature_df.iloc[:, :feature_num]
            
            eco_mu = summary_df['Eco_Mode'].values[:feature_num]
            
            pred_count = (all_feature.loc[species_id].values == eco_mu).sum()
            ref_count = (all_feature.loc[ref_list,:].values == eco_mu).sum(1)
            ref_max = ref_count.max()
            
            if pred_count < int(0.07*feature_num):
                pred_count = 0            
            score = (0.01+ pred_count) / (0.01+ ref_max)
            score = min(score, 10) 
            prob = score / (1+score) - 0.01 ## 要求更强的概率
            prob_list.append(prob)
    
    return (species_id, prob_list)

In [169]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import get_cmap

dir_path = '/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/103_leave/ZWS'
meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col=0)
meta_df = meta_df.loc[meta_df['label'].values != 2,:]
key_species_list = meta_df.index.to_list()

print(f"总共有 {len(key_species_list)} 个物种需要处理")
print(f"使用 {MAX_PROCESSES} 个进程并行处理")

top_k = 501  # 特征数量范围 1-1000
res_dic = {}

with Pool(processes=MAX_PROCESSES) as pool:
    process_func = partial(process_species, top_k=top_k)
    for species_id, prob_list in tqdm(pool.imap_unordered(process_func, key_species_list), 
                                       total=len(key_species_list), 
                                       desc="Processing species"):
        res_dic[species_id] = prob_list
        
pred_dic = {}
for ele in res_dic: 
    pred_dic[ele] = (np.array(res_dic[ele]).mean() > 0.5).astype(int)
pred_res = pd.Series(pred_dic) 
species_label = meta_df.loc[pred_res.index, 'label'] 

pred_res = pd.DataFrame([pred_res, species_label], index = ['pred', 'label']).T
print(f"acc {(pred_res['pred'] == pred_res['label']).sum()/pred_res.shape[0]}")

err_case = pred_res.index[pred_res['pred'] != pred_res['label']]
print(err_case)
meta_df.loc[err_case,]

总共有 104 个物种需要处理
使用 64 个进程并行处理


Processing species: 100%|██████████| 104/104 [00:02<00:00, 50.69it/s]


acc 0.9711538461538461
Index(['Chrysochloris_asiatica', 'Condylura_cristata', 'Physeter_catodon'], dtype='object')


,split,label,order,order_scientific,species_chinese,order_chinese,order_chinese_new
Chrysochloris_asiatica,control_1,0.0,madaowei,Afrosoricida,开普金鼹,非洲鼩目,非洲兽目
Condylura_cristata,diurnal,0.0,other,Eulipotyphla,星鼻鼹,真盲缺目,真盲缺目
Physeter_catodon,pair_2,1.0,whale,Cetacea,抹香鲸,鲸目,鲸目


### local RF + 冷启动 预测

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from multiprocessing import Pool
from functools import partial
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

# 设置最大进程数为 64
MAX_PROCESSES = 64

def process_species(species_id, top_k):
    """
    使用 RandomForest 随机森林模型计算物种的预测概率，包含冷启动策略
    返回 (species_id, prob_list) 元组，prob_list 包含每个 feature_num 下预测为 label=1 的概率
    """

    dir_path = f'/home/rsun@ZHANGroup.local/hqy_new/phd_thesis/103_leave/{species_id}'
    feature_df = pd.read_csv(os.path.join(dir_path, 'df_feature.csv'), index_col = 0)
    meta_df = pd.read_csv(os.path.join(dir_path, 'df_meta.csv'), index_col = 0)
    summary_df = pd.read_csv(os.path.join(dir_path, 'df_summary.csv'), index_col = 0)

    # if species_id == 'Sorex_araneus':
    # score_map = {5: 1, 4: 0.9, 3: 0.8, 2: 0.5, 1:0.1,0: 0}
    ## 重新打分

    score_map_4 = {4: 1, 3:0.95, 2: 0.90, 1:0.1, 0:0}
    score_map_5 = {5: 1, 4: 0.95, 3: 0.75, 2: 0.5, 1:0.1,0: 0}

    if summary_df.iloc[:,:5].sum(axis = 0).min() == 0:
        summary_df.loc[:,'cover_score'] = summary_df.loc[:,'eco_cover'].map(score_map_4)
    else: 
        summary_df.loc[:,'cover_score'] = summary_df.loc[:,'eco_cover'].map(score_map_5)
    summary_df.loc[:,'score'] = summary_df.loc[:,'cover_score'] * summary_df.loc[:,'NMI']

    summary_df = summary_df.sort_values(by = 'score', ascending=False)
    feature_df = feature_df.loc[:,summary_df.index]
    
    assert (meta_df.index == feature_df.index).all()
    assert (summary_df.index == feature_df.columns).all()

    species_order = meta_df.loc[species_id,'order_chinese_new']
    idx = summary_df['Eco_Mode'].values != '-'
    feature_df = feature_df.loc[:,idx]
    summary_df = summary_df.loc[idx,:]


    ### 构建 ref_list（训练集物种）
    if species_order == '鲸目':
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'偶蹄目']), :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '翼手目':
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'啮齿目']), :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)

    elif species_order == '真盲缺目':
        ref_list = ['Condylura_cristata',"Ceratotherium_simum_simum","Equus_asinus","Equus_quagga",]
        if species_id in ref_list:
            ref_list.remove(species_id)
    # elif species_order == '真盲缺目':
    #     tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].isin([species_order,'真盲缺目']), :]
    #     ref_list = tmp_meta_df.index.to_list()
    #     if species_id in ref_list:
    #         ref_list.remove(species_id)

    elif species_order == '啮齿目': 
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].values == species_order, :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)
            if species_id != 'ZWS':
                ref_list.remove('ZWS') # 啮齿目中唯一的非回声物种

    elif species_order == '攀鼩目':
        ref_list = ['Propithecus_coquereli','Gorilla_gorilla_gorilla', 'Pan_paniscus','Pan_troglodytes','Marmota_monax']
    elif species_order == '兔形目':
        ref_list = ['Propithecus_coquereli','Gorilla_gorilla_gorilla', 'Pan_paniscus','Ochotona_princeps', 'Ochotona_curzoniae']
        if species_id in ref_list:
            ref_list.remove(species_id)
    else:
        # 其他目：食肉目 奇蹄目 偶蹄目 非洲兽目 灵长目
        tmp_meta_df = meta_df.loc[meta_df['order_chinese_new'].values == species_order, :]
        tmp_meta_df = tmp_meta_df.loc[tmp_meta_df['label'].values == 0, :]
        ref_list = tmp_meta_df.index.to_list()
        if species_id in ref_list:
            ref_list.remove(species_id)
    
    train_species = ref_list.copy()
    
    prob_list = []
    '''鲸鱼 翼手使用机器学习模型,使用top10 feature'''
    if species_order in ['翼手目','鲸目',]:
        for feature_num in range(1, 11):
            train_feature = feature_df.loc[train_species, :].iloc[:, :feature_num]
            train_label   = meta_df.loc[train_species, 'label'].values
            target_feature = feature_df.loc[[species_id], :].iloc[:, :feature_num]
            
            # ---- Encoding：使用 OrdinalEncoder 转换为整数 ----
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoder.fit(train_feature)
            
            X_train = encoder.transform(train_feature)
            X_target = encoder.transform(target_feature)
            
            # RandomForest 可以处理浮点数，但为了保持一致性，将 -1（未知）替换为 0
            X_train_rf = np.where(X_train == -1, 0, X_train)
            X_target_rf = np.where(X_target == -1, 0, X_target)

            model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)
            model.fit(X_train_rf, train_label)
            prob_pred = model.predict_proba(X_target_rf)[0, 1]  # 取 label=1 的概率
            prob_list.append(prob_pred)
    else:
        prob_list = []
        for feature_num in range(1, top_k):
            all_feature = feature_df.iloc[:, :feature_num]
            
            eco_mu = summary_df['Eco_Mode'].values[:feature_num]
            
            pred_count = (all_feature.loc[species_id].values == eco_mu).sum()
            ref_count = (all_feature.loc[ref_list,:].values == eco_mu).sum(1)
            ref_max = ref_count.max()
            
            if pred_count < int(0.07*feature_num):
                pred_count = 0            
            score = (0.01+ pred_count) / (0.01+ ref_max)
            score = min(score, 10) 
            prob = score / (1+score) - 0.01 ## 要求更强的概率
            prob_list.append(prob)
    
    return (species_id, prob_list)